In [16]:
from Declare4Py.ProcessModels.DeclareModel import DeclareModel
from Declare4Py.ProcessMiningTasks.Discovery.DeclareMiner import DeclareMiner
from Declare4Py.D4PyEventLog import D4PyEventLog
from Declare4Py.ProcessModels.DeclareModel import DeclareModelTemplate

In [17]:
log_path = r"D:\LTNcoder\.out\eventlogs\bpic12-0.3-1.xes"

In [18]:
event_log = D4PyEventLog(case_name="case:concept:name")
event_log.parse_xes_log(log_path)

parsing log, completed traces ::   0%|          | 0/6000 [00:00<?, ?it/s]

In [19]:
discovery = DeclareMiner(log=event_log, consider_vacuity=False, min_support=0.05, itemsets_support=0.05, max_declare_cardinality=1)
declare_model: DeclareModel = discovery.run()
print(f"Total constraints discovered: {len(declare_model.serialized_constraints)}")
model_constraints = declare_model.get_decl_model_constraints()
# print("Model constraints:")
# print("-----------------")
# for idx, constr in enumerate(model_constraints):
#     print(idx, constr)


Computing discovery ...
Total constraints discovered: 961


In [20]:
declare_model.to_file("bpic12-0.3-1.decl")

In [21]:
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareAnalyzer import MPDeclareAnalyzer
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareResultsBrowser import MPDeclareResultsBrowser

basic_checker = MPDeclareAnalyzer(log=event_log, declare_model=declare_model, consider_vacuity=False)
conf_check_res: MPDeclareResultsBrowser = basic_checker.run()

In [22]:
import pickle

# Save the conformance checking results to disk to avoid recalculating
def save_conformance_results(conf_check_res, filename='bpic12_5000_conformance_results.pkl'):
    with open(filename, 'wb') as f:
        pickle.dump(conf_check_res, f)
    print(f"Conformance checking results saved to {filename}")

# Load the conformance checking results from disk
def load_conformance_results(filename='bpic12_5000_conformance_results.pkl'):
    try:
        with open(filename, 'rb') as f:
            conf_check_res = pickle.load(f)
        print(f"Conformance checking results loaded from {filename}")
        return conf_check_res
    except FileNotFoundError:
        print(f"File {filename} not found. Run conformance checking first.")
        return None

# Save the current conformance results
save_conformance_results(conf_check_res)

Conformance checking results saved to bpic12_5000_conformance_results.pkl


In [23]:
conf_check_res = load_conformance_results()


Conformance checking results loaded from bpic12_5000_conformance_results.pkl


In [24]:
conf_check_df =  conf_check_res.get_metric(metric="state")
# display(conf_check_df)

In [25]:
summary_df = conf_check_df.apply(lambda col: col.value_counts()).fillna(0).astype(int)
summary_df = summary_df.reindex([0, 1])
summary_df = summary_df / len(conf_check_df)
summary_df = summary_df.T
summary_df = summary_df.sort_values(by=1, ascending=False)
display(summary_df)

,0,1
"Choice[A_SUBMITTED+COMPLETE, A_DECLINED+COMPLETE] | |",0.000500,0.999500
"Choice[A_DECLINED+COMPLETE, A_SUBMITTED+COMPLETE] | |",0.000500,0.999500
"Choice[A_DECLINED+COMPLETE, A_PARTLYSUBMITTED+COMPLETE] | |",0.000667,0.999333
"Choice[A_PARTLYSUBMITTED+COMPLETE, A_DECLINED+COMPLETE] | |",0.000667,0.999333
"Choice[W_Completeren aanvraag+COMPLETE, A_DECLINED+COMPLETE] | |",0.003500,0.996500
...,...,...
"Alternate Response[W_Completeren aanvraag+START, A_DECLINED+COMPLETE] | |",0.942500,0.057500
"Alternate Response[W_Afhandelen leads+COMPLETE, A_DECLINED+COMPLETE] | |",0.943167,0.056833
"Chain Response[W_Completeren aanvraag+START, A_DECLINED+COMPLETE] | |",0.944833,0.055167
"Precedence[W_Completeren aanvraag+COMPLETE, A_DECLINED+COMPLETE] | |",0.947333,0.052667


In [26]:
# conf_check_df.value_counts("End[Activity B] | |")

In [27]:
import pandas as pd

In [28]:
activated_df = conf_check_res.get_metric(metric="num_activations").fillna(0)
satisfied_df = conf_check_res.get_metric(metric="state").fillna(0)
import pandas as pd
import numpy as np

# Support = how often constraint is satisfied
support = satisfied_df.sum(axis=0) / len(satisfied_df)

# Activation rate = how often constraint is activated
activated_counts = activated_df.sum(axis=0)
activation_rate = activated_counts / len(activated_df)

# Satisfied counts
satisfied_counts = satisfied_df.sum(axis=0)

# Confidence = P(satisfied | activated), safe division
confidence = np.where(
    activated_counts != 0,
    satisfied_counts / activated_counts,
    0
)
confidence = pd.Series(confidence, index=activated_counts.index)

# Combine into a DataFrame
metrics_df = pd.DataFrame({
    'support': support,
    'confidence': confidence,
    'activation_rate': activation_rate
})

C:\Users\devas\AppData\Local\Temp\ipykernel_18788\3686773640.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  activated_df = conf_check_res.get_metric(metric="num_activations").fillna(0)


In [ ]:
metrics_df

,support,confidence,activation_rate
Existence1[A_SUBMITTED+COMPLETE] | |,0.976500,0.000000,0.000000
Exactly1[A_SUBMITTED+COMPLETE] | |,0.948333,0.000000,0.000000
Init[A_SUBMITTED+COMPLETE] | |,0.976500,0.000000,0.000000
Existence1[A_PARTLYSUBMITTED+COMPLETE] | |,0.968500,0.000000,0.000000
Exactly1[A_PARTLYSUBMITTED+COMPLETE] | |,0.929000,0.000000,0.000000
...,...,...,...
"Not Precedence[W_Afhandelen leads+SCHEDULE, A_DECLINED+COMPLETE] | |",0.549833,0.579789,0.948333
"Not Chain Response[A_DECLINED+COMPLETE, W_Afhandelen leads+SCHEDULE] | |",0.918833,0.968893,0.948333
"Not Chain Response[W_Afhandelen leads+SCHEDULE, A_DECLINED+COMPLETE] | |",0.388167,0.974069,0.398500
"Not Chain Precedence[A_DECLINED+COMPLETE, W_Afhandelen leads+SCHEDULE] | |",0.389500,0.977415,0.398500


: 